# Notebook 31 — Latency validation: analysis

**Author:** Sunita Sahu · **Revision item:** A5 (iii) latency validation (decision D10) · CPU only

Reads the per-model results of notebooks 29 (18 transformers-native models) and 30 (Molmo-7B-D, DeepSeek-VL2-Small) and answers five questions:

1. Does the cost model agree with an op-level FLOP count on the **full-depth released** checkpoints?
2. Which quantity tracks measured **prefill latency** across models: the predicted FLOPs, the parameter count, or the visual-token count?
3. Does the predicted **vision share of FLOPs** match the measured vision share of prefill time?
4. Within each model, does latency scale with the predicted FLOPs from 448 × 448 to 1024 × 1024?
5. What sets **decode** cost per token?
6. Is there a **per-pass floor** below which latency carries no information about FLOPs (decision log N28)? This section also counts the operator calls of one decode step on the CPU (needs transformers 4.57.6, the release of notebook 29) and reads notebook 33's counts for Gemma 4.

**Inputs.** Laptop: unzip the `ThreadB_GPU_outputs_*.zip` archives and copy `outputs/results/*.json` into `code_active/outputs/measurements/gpu_latency/results/` (keep `logs/`, `tables/` and `environment/` next to it in `gpu_latency/`). Colab: nothing to copy; the notebook reads `MyDrive/ThreadB_GPU/outputs/results/`.

**Outputs** (`nb31_*`): run overview, full-depth FLOP validation, latency table, proxy correlations, component shares, input scaling, decode table, the per-pass floor (prefill in decode steps, operator calls per decode step, decode against calls), and a plain-text summary of the numbers.

All correlations are Spearman ρ with two-sided permutation p-values (`correlation_grid.spearman_permutation_test`: exact up to n = 10, 20,000 Monte Carlo orderings above).

---
## Settings and setup

In [ ]:
INCLUDE_DRY_RUN_RESULTS = False   # True only to test this notebook on CPU dry-run files
RESULTS_DIR_OVERRIDE = None       # a folder of result JSON files, if not the default location
# Results known to be invalid, with the reason (printed, never silently dropped), e.g.
# {'model name': 'reason'}. Empty since InstructBLIP was re-run with the fixed processor on 2026-10-03
# (decision log N20; the first result is kept in gpu_latency/superseded/).
EXCLUDED_RESULTS = {}
# N28: count the operator calls of one decode step (CPU, about 10 minutes; needs the transformers release
# of notebook 29). False, or another release, re-uses the saved nb31_operator_calls.csv.
COUNT_OPERATOR_CALLS = True

In [ ]:
import os, sys, json, zipfile
from pathlib import Path
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_FOLDER = Path('/content/drive/MyDrive/ThreadB_GPU')
    CODE_DIR = Path('/content/threadb_code')
    with zipfile.ZipFile(PROJECT_FOLDER / 'ThreadB_GPU_bundle.zip') as archive:
        archive.extractall(CODE_DIR)
    sys.path.insert(0, str(CODE_DIR))
    RESULTS_DIR = PROJECT_FOLDER / 'outputs' / 'results'
    OUTPUT_DIR = PROJECT_FOLDER / 'outputs' / 'tables'
    LOG_DIR = PROJECT_FOLDER / 'outputs' / 'logs'
else:
    NOTEBOOK_DIR = Path.cwd()
    CODE_ACTIVE = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
    sys.path.insert(0, str(CODE_ACTIVE))
    RESULTS_DIR = CODE_ACTIVE / 'outputs' / 'measurements' / 'gpu_latency' / 'results'
    OUTPUT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
    LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'
RESULTS_DIR = Path(os.environ.get('THREADB_GPU_RESULTS') or RESULTS_DIR_OVERRIDE or RESULTS_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from vlm_audit.logging_utils import ExperimentLogger, SessionTranscript
from vlm_audit.latency_analysis import (TOKEN_BUDGET_MODELS, component_shares, decode_table,
                                       decode_versus_operator_calls, diagnostics_markdown, encoder_time_versus_flops,
                                       exclude_results, executed_versus_intrinsic, findings_markdown,
                                       floor_markdown, floor_summary, flop_validation_table, gpu_names,
                                       input_scaling, latency_rank_changes, latency_table,
                                       library_release_by_model, llm_based, load_results,
                                       matched_operator_counts, prefill_floor_table, prefill_in_decode_tokens,
                                       proxy_correlations, run_overview)
if globals().get('transcript') is not None:
    transcript.close()
transcript = SessionTranscript(LOG_DIR, '31_latency_analysis')
logger = ExperimentLogger(LOG_DIR, '31_latency_analysis')
print('results from:', RESULTS_DIR)
print('tables to   :', OUTPUT_DIR)

---
## 1. Results present, their status and the GPU they ran on

In [ ]:
results = load_results(RESULTS_DIR, include_dry_run=INCLUDE_DRY_RUN_RESULTS)
if not results:
    raise FileNotFoundError(f'No results in {RESULTS_DIR}. Run notebooks 29 and 30 first (or copy their outputs here).')
overview = run_overview(results)
overview.to_csv(OUTPUT_DIR / 'nb31_run_overview.csv', index=False)   # every result, excluded ones included
results = exclude_results(results, EXCLUDED_RESULTS)
GPUS = gpu_names(results)
logger.info(f'{len(results)} results; GPUs {GPUS}')
if len(GPUS) > 1:
    print(f'WARNING: results come from {len(GPUS)} GPU types {GPUS}; latencies are only comparable within one GPU. '
          'Re-run the minority models on the majority GPU before using the latency tables.')
print(f'{len(results)} models; GPUs: {GPUS}')
overview

---
## 2. Cost model against the full-depth op-level FLOP count

Notebook 27 validated the cost model on reduced-depth copies of 18 architectures (largest difference 0.008%). Here the same comparison runs on the released, full-depth checkpoints, on the executed input (padding tiles and padded sub-images included), and for the first time on Molmo-7B-D and DeepSeek-VL2-Small.

In [ ]:
validation = flop_validation_table(results)
validation.to_csv(OUTPUT_DIR / 'nb31_flop_validation_full_depth.csv', index=False)
components_only = validation[validation.component != 'total']
if not components_only.empty:
    worst = components_only.loc[components_only.relative_difference_pct.abs().idxmax()]
    message = (f'{len(components_only)} component comparisons; largest |difference| '
               f'{abs(worst.relative_difference_pct):.4f}% ({worst.model}, {worst.input}, {worst.component})')
    print(message); logger.metric('flop_validation_largest_abs_difference_pct', abs(worst.relative_difference_pct))
validation.pivot_table(index=['model', 'input'], columns='component', values='relative_difference_pct').round(4)

---
## 3. Latency table (1024 × 1024 is the primary input)

In [ ]:
table = latency_table(results)
table.to_csv(OUTPUT_DIR / 'nb31_latency_table.csv', index=False)
columns = ['model', 'input', 'language_visual_tokens', 'predicted_total_tflops', 'prefill_ms_median', 'prefill_ms_q1',
           'prefill_ms_q3', 'achieved_tflops_per_second', 'vision_time_share_pct', 'predicted_vision_share_pct',
           'decode_ms_per_token', 'weights_resident_gb', 'prefill_activation_peak_gb', 'elementwise_share_pct']
table[columns].sort_values(['input', 'prefill_ms_median']).round(2)

---
## 4. Which quantity tracks prefill latency?

The submitted text (Secs. 5.3 and 6.1) says parameter counts are a poor proxy for forward cost and that the visual tokens entering the language model, times its size, set the cost. This is the test against measured time.

In [ ]:
proxies = proxy_correlations(table)
proxies.to_csv(OUTPUT_DIR / 'nb31_latency_proxies.csv', index=False)
for _, row in proxies.iterrows():
    logger.metric(f"rho_{row['input']}_{row['proxy']}", round(row['spearman_rho'], 4))
proxies.round(4)

---
## 5. Vision share of prefill time against the predicted vision share of FLOPs

In [ ]:
shares = component_shares(table)
shares.to_csv(OUTPUT_DIR / 'nb31_component_shares.csv', index=False)
shares.round(3)

---
## 6. Within-model scaling from 448 × 448 to 1024 × 1024

In [ ]:
scaling = input_scaling(table)
scaling['per_model'].to_csv(OUTPUT_DIR / 'nb31_input_scaling.csv', index=False)
print({key: value for key, value in scaling.items() if key != 'per_model'})
scaling['per_model'][['model', 'predicted_flop_ratio', 'latency_ratio', 'decode_ratio']].round(3)

---
## 7. Decode cost per token

In [ ]:
decode = decode_table(table)
decode['per_model'].to_csv(OUTPUT_DIR / 'nb31_decode.csv', index=False)
print({key: value for key, value in decode.items() if key != 'per_model'})
decode['per_model'].sort_values('decode_ms_per_token').round(3)

---
## 8. Diagnostics

Added after the first GPU run. (a) The proxy test on the LLM-based models only, so that a proxy cannot win merely by separating sub-billion encoders from 7B language models. (b) Executed versus intrinsic FLOPs: the padding reference implementations run. (c) Whether the cheaper model of a pair changes with the input size. (d) The vision encoder's share of prefill time against its share of FLOPs, and the throughput each component reaches. (e) Prefill cost in units of decode steps.

In [ ]:
llm_proxies = proxy_correlations(llm_based(table))
llm_proxies.to_csv(OUTPUT_DIR / 'nb31_latency_proxies_llm_based.csv', index=False)
llm_proxies.round(4)

In [ ]:
executed = executed_versus_intrinsic(table)
executed.to_csv(OUTPUT_DIR / 'nb31_executed_versus_intrinsic.csv', index=False)
executed.round(2)

In [ ]:
rank_changes = latency_rank_changes(table)
rank_changes['per_model'].to_csv(OUTPUT_DIR / 'nb31_latency_by_input.csv', index=False)
rank_changes['flipped_pairs'].to_csv(OUTPUT_DIR / 'nb31_latency_order_flips.csv', index=False)
print(f"Kendall tau {rank_changes['kendall_tau']:.2f}; {len(rank_changes['flipped_pairs'])} of {rank_changes['pairs']} pairs change order")
rank_changes['flipped_pairs'].round(1)

In [ ]:
encoder = encoder_time_versus_flops(table)
encoder.to_csv(OUTPUT_DIR / 'nb31_encoder_time_versus_flops.csv', index=False)
encoder.round(2)

In [ ]:
prefill_decode = prefill_in_decode_tokens(table)
prefill_decode.to_csv(OUTPUT_DIR / 'nb31_prefill_in_decode_tokens.csv', index=False)
prefill_decode.round(1)

---
## 9. The per-pass floor (decision log N28)

At batch size one a reference implementation issues one GPU kernel per operator call, and the host issues them one at a time. A pass therefore cannot be faster than its operator calls allow, whatever its FLOPs. Three parts:

(a) **Prefill against the floor.** A decode step is one pass over one token, so it measures the floor. The language model's prefill time is expressed in decode steps against the number of tokens it sees. Mixture-of-experts models are shown apart: a prefill over many tokens reaches most experts, a decode step only a few.

(b) **Decode against operator calls.** `operator_dispatch.count_decode_step` counts the kernel-launching calls of one decode step on the reduced-depth real architecture, with the library's default attention and expert kernels, and scales them to full depth by layer type. Each model is counted under the transformers release of its GPU measurement: 4.57.6 here, 5.18.0 for Gemma 4 in notebook 33. Qwen3-VL-8B is counted under both. Molmo-7B-D and DeepSeek-VL2-Small (remote code, 4.46.3) are not counted.

(c) The prefill correlations of section 4 without the two token-budget models, which sit on the floor at every image size.

In [ ]:
floor = prefill_floor_table(table)
floor.to_csv(OUTPUT_DIR / 'nb31_prefill_floor.csv', index=False)
floor_overview = floor_summary(floor)
print({key: value for key, value in floor_overview.items() if key != 'mixture_of_experts_rows'})
floor.round(2)

In [ ]:
import transformers
from transformers.utils import logging as transformers_logging
from vlm_audit.operator_dispatch import BRIDGE_MODEL, TRANSFORMERS4_DECODE_MODELS, decode_step_table

native_release = library_release_by_model(results).get('LLaVA-1.5-7B')
counts_path = OUTPUT_DIR / 'nb31_operator_calls.csv'
if COUNT_OPERATOR_CALLS and transformers.__version__ == native_release:
    transformers_logging.set_verbosity_error()
    operator_counts = decode_step_table(TRANSFORMERS4_DECODE_MODELS, logger=logger)
    operator_counts.to_csv(counts_path, index=False)
elif counts_path.exists():
    operator_counts = pd.read_csv(counts_path)
    print(f'transformers {transformers.__version__} (GPU run: {native_release}): re-using {counts_path.name}')
else:
    operator_counts = pd.DataFrame(columns=['model', 'status', 'transformers_version', 'calls_per_decode_step'])
    print(f'No operator counts: run this notebook with transformers {native_release} to count them.')
transformers5_counts_path = OUTPUT_DIR / 'nb33_operator_calls.csv'
if transformers5_counts_path.exists():
    operator_counts = pd.concat([operator_counts, pd.read_csv(transformers5_counts_path)], ignore_index=True)
else:
    print('nb33_operator_calls.csv not found: Gemma 4 is not counted (run notebook 33 first).')
matched_counts = matched_operator_counts(operator_counts, results)
bridge = operator_counts[(operator_counts['model'] == BRIDGE_MODEL) & (operator_counts['status'] == 'complete')]
dispatch = decode_versus_operator_calls(table, matched_counts)
dispatch['per_model'].to_csv(OUTPUT_DIR / 'nb31_decode_versus_operator_calls.csv', index=False)
print({key: value for key, value in dispatch.items() if key != 'per_model'})
dispatch['per_model'].round(2)

In [ ]:
budget_free_proxies = proxy_correlations(table[~table['model'].isin(TOKEN_BUDGET_MODELS)])
budget_free_proxies = budget_free_proxies[budget_free_proxies['proxy'].isin(['predicted_total_gflops',
                                                                            'parameter_count'])]
budget_free_proxies.round(4)

---
## 10. Summary of the numbers and closing the logs

In [ ]:
summary_text = findings_markdown(results, validation, table, proxies, shares, scaling, decode)
summary_text += diagnostics_markdown(llm_proxies, executed, rank_changes, encoder, prefill_decode, EXCLUDED_RESULTS)
summary_text += floor_markdown(floor_overview, dispatch, budget_free_proxies, bridge)
(OUTPUT_DIR / 'nb31_latency_findings.md').write_text(summary_text, encoding='utf-8')
print(summary_text)
logger.close()
transcript.close()
transcript = None